# 15 - Features, fitted transformations and the model-data release
Builds the model features from `docs/feature_plan.xlsx`, fits the transformations on the training rows only, and writes model-data release v2.

| Part | What it does | Fitted? |
|---|---|---|
| 1 | Load the matches, splits, venues and weather | - |
| 2 | Fixed per-row features (rank gap, age gap, entry flags, categories) | no |
| 3 | History features, from earlier tournaments only (form, surface form, earlier matches, head-to-head) | no |
| 4 | Weather over the match-day window, venue elevation | no |
| 5 | The feature table, stored in BigQuery | - |
| 6 | Independent check: history features recomputed a second way for a random sample | - |
| 7 | Fitted transformations (imputation, scaling, one-hot encoding), **fitted on train only** | yes |
| 8 | Checks: leakage, train-only fitting, columns and row counts, same input gives the same output | - |
| 9 | Release v2 (Parquet per split), the fitted transformers, manifest | - |
| 10 | `docs/feature_spec.yaml`, written from the same feature list the code uses | - |

Test and holdout rows are loaded only so that their features can be computed. Nothing is fitted on them or examined here.
Run after 10_split and 08_weather_raw.

## Install pinned libraries
Read directly from the Github repo

In [ ]:
# Read from a git tag rather than from main, so a later re-run installs exactly the same versions.
CODE_VERSION = "code-v9"      # the git tag of the code being run: create it on GitHub before running
REPO = "katiedevane/tennis-project"
REQUIREMENTS_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/requirements.txt"
!pip install -q -r $REQUIREMENTS_URL

# Shared provenance helpers (code version, environment, index of data versions), from the same tag
PROVENANCE_URL = f"https://raw.githubusercontent.com/{REPO}/{CODE_VERSION}/provenance.py"
!wget -q -O provenance.py $PROVENANCE_URL

## Authenticate and setup
Enable Notebook access for both secrets in this notebook.

In [ ]:
import os
from google.colab import auth, userdata

auth.authenticate_user()

PROJECT_ID = userdata.get("TENNIS_PROJECT_ID")
BUCKET_NAME = userdata.get("TENNIS_BUCKET_NAME")
REGION = "EU"        # the bucket and the BigQuery datasets must share this location
DATASETS = ["tennis_raw", "tennis_clean", "tennis_features"]
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

assert PROJECT_ID and BUCKET_NAME, "A secret is empty: check its name and Notebook access"
print("Settings loaded from Colab Secrets.")

## Record the code version and set the random seed
The git tag, commit, Python environment and random seed of this run are added to every manifest this notebook writes (see provenance.py).
`SEED` is the one project-wide seed: `start_run` applies it, and it should be passed wherever a function takes a seed (e.g. `random_state=SEED`).

In [ ]:
from provenance import start_run, register, table_version, sha256, utc_stamp, SEED

NOTEBOOK = "15_features"
RUN_INFO = start_run(f"https://github.com/{REPO}", CODE_VERSION, NOTEBOOK)

## Helpers
scikit-learn is not yet in requirements.txt: the version printed below should be added there (and a new code tag made).

In [ ]:
import gzip, hashlib, io, json, os, random
import numpy as np
import pandas as pd
import sklearn, joblib, yaml
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from google.cloud import bigquery, storage

bq = bigquery.Client(project=PROJECT_ID, location=REGION)
bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)

def run(sql):
    config = bigquery.QueryJobConfig(maximum_bytes_billed=1_000_000_000)   # safety cap: 1 GB per query
    return bq.query(sql, job_config=config).to_dataframe()

pd.set_option("display.max_rows", 200, "display.max_columns", 100, "display.width", 200)
print("scikit-learn", sklearn.__version__, "| joblib", joblib.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

AB = f"{PROJECT_ID}.tennis_clean.matches_ab"
SPLITS = f"{PROJECT_ID}.tennis_features.match_splits"
EDITIONS = f"{PROJECT_ID}.tennis_clean.tournament_editions"
FEATURE_TABLE = f"{PROJECT_ID}.tennis_features.match_features_v1"
UNRANKED_RANK = 2000          # fixed fill for unranked players: worse than any rank in the data (max about 1,560)
FORM_PRIOR_MATCHES = 5        # smoothing strength: 5 imaginary matches, half won
FORM_WINDOW = "365D"          # recent form: the 365 days before the tournament starts

## The feature list (one source for the code and the specification)
Every feature, its group (which decides its fitted transformation), and its documentation. `docs/feature_spec.yaml` is written from this list in Part 10, so the code and the specification cannot disagree.

In [ ]:
FEATURES = [
    # name, group, dtype, definition, inputs, missing-value rule
    ("log_rank_ratio", "difference", "FLOAT", "log(B rank) - log(A rank); > 0 when A is better ranked",
     ["a_rank", "b_rank"], f"unranked: fixed rank {UNRANKED_RANK} (worse than any rank) + unranked flag"),
    ("age_diff", "difference", "FLOAT", "A age - B age, years", ["a_age", "b_age"], "training median (under 0.1% missing)"),
    ("form_diff_52w", "difference", "FLOAT",
     f"A minus B: smoothed win rate in the {FORM_WINDOW} before the tournament start, (wins + {FORM_PRIOR_MATCHES / 2}) / (matches + {FORM_PRIOR_MATCHES})",
     ["a_id", "b_id", "tourney_date", "a_won"], "none: smoothing gives a value for every player"),
    ("surface_form_diff_52w", "difference", "FLOAT", "As form_diff_52w, on the same surface only",
     ["a_id", "b_id", "surface", "tourney_date", "a_won"], "none (smoothed)"),
    ("prior_matches_log_diff", "difference", "FLOAT", "log(1 + A earlier matches) - log(1 + B earlier matches), all earlier tournaments",
     ["a_id", "b_id", "tourney_date"], "none (0 earlier matches is a value)"),
    ("h2h_diff", "difference", "INTEGER", "A wins minus B wins in their earlier meetings", ["a_id", "b_id", "tourney_date", "a_won"],
     "none (no earlier meeting = 0)"),
    ("temp", "weather", "FLOAT", "Daytime (10:00-20:00 local) temperature averaged over the possible match days, deg C",
     ["weather file", "match_date_min", "match_date_max"], "training median (none expected)"),
    ("humidity", "weather", "FLOAT", "As temp: relative humidity, %", ["weather file"], "training median (none expected)"),
    ("wind", "weather", "FLOAT", "As temp: wind speed, km/h", ["weather file"], "training median (none expected)"),
    ("rain_mm", "weather_log", "FLOAT", "Daytime precipitation, mm per day, averaged over the possible match days", ["weather file"],
     "training median (none expected)"),
    ("elevation_m", "weather_log", "FLOAT", "Venue elevation, metres", ["venue_locations"], "training median (none expected)"),
    ("a_unranked", "flag", "INTEGER", "1 if player A has no ranking", ["a_rank"], "none"),
    ("b_unranked", "flag", "INTEGER", "1 if player B has no ranking", ["b_rank"], "none"),
    ("a_entry_q", "flag", "INTEGER", "1 if player A entered as a qualifier", ["a_entry"], "empty entry = direct entry (0)"),
    ("a_entry_wc", "flag", "INTEGER", "1 if player A entered with a wildcard", ["a_entry"], "empty entry = direct entry (0)"),
    ("a_entry_ll", "flag", "INTEGER", "1 if player A entered as a lucky loser", ["a_entry"], "empty entry = direct entry (0)"),
    ("b_entry_q", "flag", "INTEGER", "1 if player B entered as a qualifier", ["b_entry"], "empty entry = direct entry (0)"),
    ("b_entry_wc", "flag", "INTEGER", "1 if player B entered with a wildcard", ["b_entry"], "empty entry = direct entry (0)"),
    ("b_entry_ll", "flag", "INTEGER", "1 if player B entered as a lucky loser", ["b_entry"], "empty entry = direct entry (0)"),
    ("surface", "category", "STRING", "hard, clay, grass, carpet", ["surface"], "none missing"),
    ("tourney_level", "category", "STRING", "G, PM, P, I", ["tourney_level"], "none missing"),
    ("round", "category", "STRING", "R128 ... F", ["round"], "none missing"),
    ("ht_diff", "ablation", "FLOAT", "A height - B height, cm. NOT in the default model: dev-only ablation",
     ["a_ht", "b_ht"], "training median, no missing flag (the meaning of missing shifts over time)"),
]
F = pd.DataFrame(FEATURES, columns=["name", "group", "dtype", "definition", "inputs", "missing"])
GROUP_TRANSFORM = {
    "difference": "imputation (training median), StandardScaler without centring: 0 stays 'equal players'",
    "weather": "imputation (training median), StandardScaler",
    "weather_log": "log(1 + x), imputation (training median), StandardScaler",
    "flag": "none (0/1, passed through)",
    "category": "OneHotEncoder, categories learned from train, unknown categories ignored",
    "ablation": "only in the ablation: imputation (training median), StandardScaler without centring",
}
cols = lambda group: F.loc[F["group"] == group, "name"].tolist()
print(F.groupby("group")["name"].apply(list).to_string())

## 1. Load the data
All splits, because history features use the whole timeline; walkovers are kept only for counting (no tennis was played, so they are not history and not modelled).

In [ ]:
df = run(f"SELECT a.*, s.split, s.modelling_ok FROM `{AB}` a JOIN `{SPLITS}` s USING (match_id)")
for c in df.columns:
    if str(df[c].dtype) == "Int64":
        df[c] = df[c].astype("float64")
for c in ["tourney_date", "match_date_min", "match_date_max"]:
    df[c] = pd.to_datetime(df[c])
df["modelling_ok"] = df["modelling_ok"].astype(bool)
m = df[df["modelling_ok"]].copy().sort_values(["tourney_date", "match_id"]).reset_index(drop=True)

assert not set(F["name"]) & {c for c in df.columns if c.startswith("post_")}, "A feature name clashes with a post-match column"
print("matches:", len(df), "| played (modelling rows):", len(m))
print(m["split"].value_counts().reindex(["warmup", "train", "dev", "test", "holdout_2026"]).to_string())

## 2. Fixed per-row features
Nothing is learned from the data: the same rule gives the same value whichever split a row is in.

In [ ]:
a_rank = m["a_rank"].fillna(UNRANKED_RANK)
b_rank = m["b_rank"].fillna(UNRANKED_RANK)
assert m[["a_rank", "b_rank"]].max().max() < UNRANKED_RANK, "A ranked player is worse than the unranked fill value"
m["log_rank_ratio"] = np.log(b_rank) - np.log(a_rank)
m["a_unranked"] = m["a_rank"].isna().astype(int)
m["b_unranked"] = m["b_rank"].isna().astype(int)
m["age_diff"] = m["a_age"] - m["b_age"]
m["ht_diff"] = m["a_ht"] - m["b_ht"]                     # ablation only
for side in ["a", "b"]:
    entry = m[f"{side}_entry"].fillna("")
    for code_, name in [("Q", "q"), ("WC", "wc"), ("LL", "ll")]:
        m[f"{side}_entry_{name}"] = (entry == code_).astype(int)

print("best_of values:", m["best_of"].value_counts(dropna=False).to_dict(), "-> constant, so not used" if m["best_of"].nunique() == 1 else "-> NOT constant: review")
print(m[["log_rank_ratio", "age_diff", "a_unranked", "b_unranked"]].describe().round(3))

## 3. History features, from earlier tournaments only
Each player's record is taken from **played matches in tournaments that started before this one**: a window that ends the day before the tournament start (`closed="left"`), so the current match, its tournament and anything later are never used.
The 2010-2015 warmup seasons provide history for the first training seasons.

In [ ]:
# One row per player per played match
long = pd.concat([
    pd.DataFrame({"match_id": m["match_id"], "id": m["a_id"], "opp": m["b_id"], "tourney_date": m["tourney_date"],
                  "surface": m["surface"], "won": m["a_won"]}),
    pd.DataFrame({"match_id": m["match_id"], "id": m["b_id"], "opp": m["a_id"], "tourney_date": m["tourney_date"],
                  "surface": m["surface"], "won": 1 - m["a_won"]}),
], ignore_index=True)

def windowed(keys):
    """Wins and matches per key in the FORM_WINDOW before each date (earlier dates only)."""
    daily = (long.groupby(keys + ["tourney_date"]).agg(w=("won", "sum"), n=("won", "size"))
                 .reset_index().sort_values(keys + ["tourney_date"]))
    rolled = (daily.set_index("tourney_date").groupby(keys)[["w", "n"]]
                   .rolling(FORM_WINDOW, closed="left").sum().reset_index())
    return rolled.fillna({"w": 0, "n": 0})

def smoothed(w, n):
    return (w + FORM_PRIOR_MATCHES / 2) / (n + FORM_PRIOR_MATCHES)

form = windowed(["id"]).assign(form=lambda d: smoothed(d["w"], d["n"]))
surface_form = windowed(["id", "surface"]).assign(sform=lambda d: smoothed(d["w"], d["n"]))

prior = long.groupby(["id", "tourney_date"]).size().rename("n").reset_index().sort_values(["id", "tourney_date"])
prior["prior"] = prior.groupby("id")["n"].cumsum() - prior["n"]             # all earlier tournaments

h2h = (long.groupby(["id", "opp", "tourney_date"])["won"].sum().rename("w").reset_index()
           .sort_values(["id", "opp", "tourney_date"]))
h2h["h2h_wins"] = h2h.groupby(["id", "opp"])["w"].cumsum() - h2h["w"]      # wins against this opponent before this date

def attach(side, table, value, keys, new):
    other = "b" if side == "a" else "a"
    rename = {"id": f"{side}_id", "opp": f"{other}_id", value: new}
    t = table[keys + ["tourney_date", value]].rename(columns=rename)
    on = [rename.get(k, k) for k in keys] + ["tourney_date"]
    return m.merge(t, on=on, how="left", validate="many_to_one")

for side in ["a", "b"]:
    m = attach(side, form, "form", ["id"], f"{side}_form")
    m = attach(side, surface_form, "sform", ["id", "surface"], f"{side}_surface_form")
    m = attach(side, prior, "prior", ["id"], f"{side}_prior")
    m = attach(side, h2h, "h2h_wins", ["id", "opp"], f"{side}_h2h_wins")

m["form_diff_52w"] = m["a_form"] - m["b_form"]
m["surface_form_diff_52w"] = m["a_surface_form"] - m["b_surface_form"]
m["prior_matches_log_diff"] = np.log1p(m["a_prior"]) - np.log1p(m["b_prior"])
m["h2h_diff"] = m["a_h2h_wins"] - m["b_h2h_wins"]
hist = ["form_diff_52w", "surface_form_diff_52w", "prior_matches_log_diff", "h2h_diff"]
assert m[hist].notna().all().all(), "A history feature is missing: every played match should have a value"
print(m[hist].describe().round(3))

## 4. Weather over the match-day window, and venue elevation
Same method as the EDA (notebook 13): daytime hours (10:00-20:00 local), averaged over the window of possible match days.

In [ ]:
names = sorted(b.name for b in bucket.list_blobs(prefix="manifests/open_meteo_archive_v2_"))
weather_manifest = json.loads(bucket.blob(names[-1]).download_as_text())
WEATHER = weather_manifest["files"][0]
assert WEATHER["file"].endswith(".jsonl.gz"), "Run 08_weather_raw_revised first (packed weather file)"
data = bucket.blob(WEATHER["file"]).download_as_bytes()
assert hashlib.sha256(data).hexdigest() == WEATHER["sha256"], "The weather file differs from its manifest"

parts = []
for line in gzip.decompress(data).splitlines():
    rec = json.loads(line)
    hourly = pd.DataFrame(rec["response"]["hourly"])
    hourly["tourney_id"] = rec["tourney_id"]
    parts.append(hourly)
hourly = pd.concat(parts, ignore_index=True)
hourly["time"] = pd.to_datetime(hourly["time"])
day = (hourly[hourly["time"].dt.hour.between(10, 20)]
       .assign(date=lambda d: d["time"].dt.normalize())
       .groupby(["tourney_id", "date"])
       .agg(temp=("temperature_2m", "mean"), humidity=("relative_humidity_2m", "mean"),
            wind=("wind_speed_10m", "mean"), rain_mm=("precipitation", "sum"))
       .reset_index())

win = m[["match_id", "tourney_id", "match_date_min", "match_date_max"]].copy()
win["days"] = (win["match_date_max"] - win["match_date_min"]).dt.days + 1
win = win.loc[win.index.repeat(win["days"])]
win["date"] = win["match_date_min"] + pd.to_timedelta(win.groupby(level=0).cumcount(), unit="D")
weather = win.merge(day, on=["tourney_id", "date"], how="left").groupby("match_id")[["temp", "humidity", "wind", "rain_mm"]].mean()
m = m.merge(weather, left_on="match_id", right_index=True, how="left")
m = m.merge(run(f"SELECT tourney_id, elevation_m FROM `{EDITIONS}`"), on="tourney_id", how="left")
print("matches with weather:", f"{m['temp'].notna().mean():.1%}", "| with elevation:", f"{m['elevation_m'].notna().mean():.1%}")

## 5. The feature table
One row per played match: identifiers, split, label, and every feature before any fitted transformation. Stored in BigQuery (replaced on every run).

In [ ]:
ID_COLS = ["match_id", "tourney_id", "tourney_date", "season", "split", "a_id", "b_id"]
LABEL = "a_won"
FEATURE_COLS = F["name"].tolist()
features = m[ID_COLS + [LABEL] + FEATURE_COLS].sort_values(["tourney_date", "match_id"]).reset_index(drop=True)
features["a_won"] = features["a_won"].astype(int)
features["tourney_date"] = features["tourney_date"].dt.date
for c in ["a_id", "b_id", "season"] + cols("flag") + ["h2h_diff"]:
    features[c] = features[c].astype(int)

bq.load_table_from_dataframe(features, FEATURE_TABLE,
                             job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE")).result()
print(FEATURE_TABLE, "| rows:", bq.get_table(FEATURE_TABLE).num_rows, "| columns:", len(features.columns))
print(features.groupby("split").size().reindex(["warmup", "train", "dev", "test", "holdout_2026"]).to_string())

## 6. Independent check of the history features
For a random sample of matches (seeded), each history feature is recomputed a second, deliberately simple way: by looping over that player's earlier matches directly. Both methods must agree exactly.

In [ ]:
random.seed(SEED)
sample = features.loc[random.sample(range(len(features)), 300)]
by_player = {pid: g.sort_values("tourney_date") for pid, g in long.groupby("id")}

def brute(pid, opp, date, surface):
    g = by_player[pid]
    earlier = g[g["tourney_date"] < date]
    recent = earlier[earlier["tourney_date"] >= date - pd.Timedelta(FORM_WINDOW)]
    rs = recent[recent["surface"] == surface]
    return (smoothed(recent["won"].sum(), len(recent)), smoothed(rs["won"].sum(), len(rs)),
            len(earlier), earlier.loc[earlier["opp"] == opp, "won"].sum())

mismatches = 0
for r in sample.itertuples():
    d = pd.Timestamp(r.tourney_date)
    a = brute(r.a_id, r.b_id, d, r.surface)
    b = brute(r.b_id, r.a_id, d, r.surface)
    expected = (a[0] - b[0], a[1] - b[1], np.log1p(a[2]) - np.log1p(b[2]), a[3] - b[3])
    got = (r.form_diff_52w, r.surface_form_diff_52w, r.prior_matches_log_diff, r.h2h_diff)
    mismatches += not np.allclose(expected, got, rtol=0, atol=1e-12)
print(f"history features recomputed for {len(sample)} random matches: {mismatches} mismatches")
assert mismatches == 0

## 7. Fitted transformations: fitted on the training rows only
One scikit-learn `ColumnTransformer` learns the imputation medians, the scaling factors and the category vocabularies **from train only**, and is then applied unchanged to dev, test and holdout.
For the final model (the split definition's refit rule: tune on dev, refit on train + dev, evaluate once on test) a second transformer is fitted on train + dev.

In [ ]:
def make_transformer(include_height=False):
    diff = cols("difference") + (cols("ablation") if include_height else [])
    return ColumnTransformer([
        ("difference", Pipeline([("impute", SimpleImputer(strategy="median")),
                                 ("scale", StandardScaler(with_mean=False))]), diff),
        ("weather", Pipeline([("impute", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), cols("weather")),
        ("weather_log", Pipeline([("log", FunctionTransformer(np.log1p, feature_names_out="one-to-one")),
                                  ("impute", SimpleImputer(strategy="median")),
                                  ("scale", StandardScaler())]), cols("weather_log")),
        ("flag", "passthrough", cols("flag")),
        ("category", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cols("category")),
    ], verbose_feature_names_out=False)

INPUT_COLS = [c for c in FEATURE_COLS if c not in cols("ablation")]
train = features[features["split"] == "train"]
train_dev = features[features["split"].isin(["train", "dev"])]

transformer = make_transformer().fit(train[INPUT_COLS])
transformer_final = make_transformer().fit(train_dev[INPUT_COLS])
OUTPUT_COLS = list(transformer.get_feature_names_out())
print("fitted on", len(train), "training rows |", len(INPUT_COLS), "input features ->", len(OUTPUT_COLS), "model columns")
print(OUTPUT_COLS)

## 8. Checks

In [ ]:
def model_input(frame, t=transformer):
    return pd.DataFrame(t.transform(frame[INPUT_COLS]), columns=OUTPUT_COLS, index=frame.index)

expected_rows = run(f"SELECT split, COUNTIF(modelling_ok) AS n FROM `{SPLITS}` GROUP BY split").set_index("split")["n"]
imp = transformer.named_transformers_["difference"].named_steps["impute"]
checks = {
    "no post-match column is a feature": not any(c.startswith("post_") for c in FEATURE_COLS),
    "history features use only earlier tournaments (Part 6: 0 mismatches)": mismatches == 0,
    "transformer fitted on exactly the training rows":
        transformer.named_transformers_["weather"].named_steps["scale"].n_samples_seen_ == len(train),
    "imputation medians equal the training medians":
        np.allclose(imp.statistics_, train[cols("difference")].median().to_numpy(), equal_nan=True),
    "category vocabulary comes from train only":
        all(set(cats) == set(train[c].dropna().unique())
            for c, cats in zip(cols("category"), transformer.named_transformers_["category"].categories_)),
    "row count per split equals the modelling rows in match_splits":
        all(len(features[features["split"] == s]) == int(expected_rows[s]) for s in expected_rows.index),
    "every split gives the same model columns":
        all(list(model_input(features[features["split"] == s]).columns) == OUTPUT_COLS
            for s in ["train", "dev", "test", "holdout_2026"]),
    "no missing values after the transformation":
        not any(model_input(features[features["split"] == s]).isna().any().any() for s in ["train", "dev", "test", "holdout_2026"]),
    "same input gives the same output (fit twice)":
        np.array_equal(make_transformer().fit(train[INPUT_COLS]).transform(train[INPUT_COLS]),
                       transformer.transform(train[INPUT_COLS])),
    "difference features keep 0 as 'equal players' (no centring)":
        np.allclose(transformer.transform(train[INPUT_COLS].iloc[:1].assign(**{c: 0.0 for c in cols("difference")}))[0][:len(cols("difference"))], 0),
}
for name, ok in checks.items():
    print("PASS " if ok else "CHECK", name)
assert all(checks.values()), "A check failed: do not write the release"

## 9. Dataset generation: release v2
One folder per split (`train`, `dev`, `test`, `holdout_2026`; warmup is history only), each with:
- `features.parquet`: identifiers, label and features **before** fitting (any model can refit from these)
- `model_input.parquet`: identifiers, label and the transformed columns from the train-fitted transformer

plus the two fitted transformers and a manifest. A release is written once: if it exists with different content, the notebook stops.

In [ ]:
RELEASE = "v2"
PREFIX = f"processed/model_data/release={RELEASE}"
MANIFEST_PREFIX = f"manifests/model_data_{RELEASE}_"
EXPORT = ["train", "dev", "test", "holdout_2026"]
os.makedirs("release", exist_ok=True)

def content_hash(frame):
    """Hash of the values (independent of the file format), so identical content is recognised."""
    return hashlib.sha256(pd.util.hash_pandas_object(frame, index=False).to_numpy().tobytes()).hexdigest()

files, data_hashes = [], {}
for split in EXPORT:
    part = features[features["split"] == split]
    outputs = {"features": part,
               "model_input": pd.concat([part[ID_COLS + [LABEL]], model_input(part)], axis=1)}
    for kind, frame in outputs.items():
        local = f"release/{split}_{kind}.parquet"
        frame.to_parquet(local, index=False)
        data_hashes[f"{split}/{kind}"] = content_hash(frame)
        files.append({"file": f"{PREFIX}/split={split}/{kind}.parquet", "local": local, "rows": len(frame),
                      "columns": len(frame.columns), "content_sha256": data_hashes[f"{split}/{kind}"]})
for name, t in [("transformer_train", transformer), ("transformer_train_dev", transformer_final)]:
    local = f"release/{name}.joblib"
    joblib.dump(t, local)
    files.append({"file": f"{PREFIX}/{name}.joblib", "local": local, "fitted_on": name.split("_", 1)[1].replace("_", " + ")})

existing = sorted(b.name for b in bucket.list_blobs(prefix=MANIFEST_PREFIX))
if existing:
    recorded = json.loads(bucket.blob(existing[0]).download_as_text())["content_sha256"]
    if recorded != data_hashes:
        raise RuntimeError(f"Release {RELEASE} exists with different content. Use a new RELEASE.")
    print(f"Release {RELEASE} already exists with identical content. Nothing written.")
    manifest_path = existing[0]
else:
    for f in files:
        blob = bucket.blob(f["file"])
        blob.upload_from_filename(f["local"], if_generation_match=0)
        f["bytes"] = os.path.getsize(f["local"])
        f["sha256"] = hashlib.sha256(open(f["local"], "rb").read()).hexdigest()
    parents = [table_version(bq, AB), table_version(bq, SPLITS), table_version(bq, FEATURE_TABLE),
               {"file": WEATHER["file"], "sha256": WEATHER["sha256"]}]
    manifest_path = f"{MANIFEST_PREFIX}{utc_stamp()}.json"
    bucket.blob(manifest_path).upload_from_string(json.dumps({
        "release": RELEASE, "contents": "pre-match features (fixed, history, weather) + fitted transformations",
        "features": FEATURE_COLS, "model_columns": OUTPUT_COLS, "label": LABEL,
        "settings": {"unranked_rank": UNRANKED_RANK, "form_prior_matches": FORM_PRIOR_MATCHES, "form_window": FORM_WINDOW},
        "files": [{k: v for k, v in f.items() if k != "local"} for f in files],
        "content_sha256": data_hashes, "parents": parents, **RUN_INFO}, indent=1, default=str),
        content_type="application/json")
    register(bucket, "processed/model_data", RELEASE, outputs=[{k: v for k, v in f.items() if k != "local"} for f in files],
             parents=parents, manifest_path=manifest_path)
    print("written:", len(files), "files under", PREFIX)
print("manifest:", manifest_path)
print(pd.DataFrame(files)[["file", "rows", "columns"]].to_string(index=False))

## 10. Feature specification
`docs/feature_spec.yaml`, written from the same feature list the code uses. Saved in the bucket and downloaded: commit it to the repo under `docs/`.

In [ ]:
spec = {
    "feature_set_version": RELEASE,
    "code_version": RUN_INFO["code_version"],
    "source_table": FEATURE_TABLE.split(".", 1)[1],
    "label": "a_won (INTEGER, 1 if player A won)",
    "rows": "played matches (walkovers excluded); splits by season (tennis_features.match_splits)",
    "random_seed": SEED,
    "fitted_on": "train (transformer_train); train + dev for the final model (transformer_train_dev)",
    "left_out": {"best_of": "constant (all best of 3)",
                 "ranking points": "duplicate rank (Spearman about -0.998)",
                 "missing-height flags": "the meaning of 'missing' shifts between train and later seasons"},
    "features": [
        {"name": r.name, "dtype": r.dtype, "group": r.group, "definition": r.definition, "inputs": r.inputs,
         "known_before_match": "yes" + (" (earlier tournaments only)" if r.name in hist else ""),
         "missing_values": r.missing, "transformation": GROUP_TRANSFORM[r.group],
         "in_default_model": r.group != "ablation"}
        for r in F.itertuples()],
}
text = yaml.safe_dump(spec, sort_keys=False, allow_unicode=True, width=120)
bucket.blob("docs/feature_spec.yaml").upload_from_string(text, content_type="text/yaml")
with open("feature_spec.yaml", "w") as f:
    f.write(text)
print(text[:1500])

from google.colab import files as colab_files
colab_files.download("feature_spec.yaml")